# Retail Intelligence — End-to-End Databricks Project
**Pipeline:** REST API → raw JSON → Bronze → Silver → EDA & quality → Gold → Power BI.

Uses synthetic public e-commerce data. Run top to bottom in a Databricks notebook.

## 1. Configuration and imports
Set `CATALOG` to a catalog where you can create schemas and a Volume.

In [ ]:
import json, time, uuid, requests
from datetime import datetime, timezone
from pyspark.sql import functions as F, types as T

API_BASE = "https://api.sondahub.com/v1/store"
CATALOG = "main"  # CHANGE THIS to your Unity Catalog catalog
BRONZE_SCHEMA = "retail_bronze"
SILVER_SCHEMA = "retail_silver"
GOLD_SCHEMA = "retail_gold"
VOLUME = "raw_api"
PAGE_SIZE = 100
COLLECTIONS = ["categories", "products", "customers", "orders", "order_items", "reviews", "warehouses", "inventory", "carts"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{VOLUME}")
RAW_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}"
print("Raw landing path:", RAW_PATH)

## 2. API ingestion — paginated REST calls
Each page is persisted as an immutable JSON file in the Unity Catalog Volume. Retries are included for transient HTTP errors.

In [ ]:
session = requests.Session()
session.headers.update({"Accept": "application/json", "User-Agent": "retail-intelligence-portfolio/1.0"})

def get_page(collection, page, limit=PAGE_SIZE, retries=3):
    url = f"{API_BASE}/{collection}"
    for attempt in range(retries):
        try:
            response = session.get(url, params={"page": page, "limit": limit}, timeout=45)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            if attempt == retries - 1: raise
            time.sleep(2 ** attempt)

ingestion_rows = []
for collection in COLLECTIONS:
    page = 1
    while True:
        payload = get_page(collection, page)
        rows = payload.get("data", []) if isinstance(payload, dict) else []
        if not rows: break
        file_name = f"{collection}_page_{page:05d}_{uuid.uuid4().hex[:8]}.json"
        dbutils.fs.put(f"{RAW_PATH}/{file_name}", json.dumps(rows, ensure_ascii=False), overwrite=False)
        ingestion_rows.append((collection, page, len(rows), file_name, datetime.now(timezone.utc).isoformat(), json.dumps(payload.get("meta", {}))))
        meta = payload.get("meta", {})
        total_pages = meta.get("totalPages") or meta.get("total_pages")
        if total_pages is not None and page >= int(total_pages): break
        if len(rows) < PAGE_SIZE: break
        page += 1
    print(f"{collection}: ingested {page} page(s)")

ingestion_schema = T.StructType([T.StructField("collection",T.StringType()),T.StructField("page",T.IntegerType()),T.StructField("record_count",T.IntegerType()),T.StructField("file_name",T.StringType()),T.StructField("ingested_at_utc",T.StringType()),T.StructField("api_meta",T.StringType())])
ingestion_df = spark.createDataFrame(ingestion_rows, ingestion_schema)
display(ingestion_df)

## 3. Bronze — raw API pages in Delta
Keep the original page payload alongside ingestion metadata for traceability and replay.

In [ ]:
bronze_rows = []
for r in ingestion_rows:
    raw_text = dbutils.fs.head(f"{RAW_PATH}/{r[3]}", 10_000_000)
    bronze_rows.append((r[0], r[1], r[2], r[3], r[4], raw_text))
bronze_schema = T.StructType([T.StructField("collection",T.StringType()),T.StructField("page",T.IntegerType()),T.StructField("record_count",T.IntegerType()),T.StructField("file_name",T.StringType()),T.StructField("ingested_at_utc",T.StringType()),T.StructField("raw_payload",T.StringType())])
bronze_df = spark.createDataFrame(bronze_rows, bronze_schema)
bronze_table = f"{CATALOG}.{BRONZE_SCHEMA}.api_pages"
bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(bronze_table)
display(spark.table(bronze_table))

## 4. Silver — parse, flatten and persist entities
The API returns arrays of records inside `data`. Struct fields are flattened recursively; array fields are retained as JSON strings so no nested information is silently discarded.

In [ ]:
def flatten_structs(df):
    while True:
        nested = [(f.name, f.dataType) for f in df.schema.fields if isinstance(f.dataType, T.StructType)]
        if not nested: return df
        name, dtype = nested[0]
        expanded = [F.col(f"`{name}`.`{child.name}`").alias(f"{name}_{child.name}") for child in dtype.fields]
        remaining = [F.col(f"`{f.name}`") for f in df.schema.fields if f.name != name]
        df = df.select(*remaining, *expanded)

silver_tables = {}
for collection in COLLECTIONS:
    paths = [f"{RAW_PATH}/{r[3]}" for r in ingestion_rows if r[0] == collection]
    if not paths: continue
    raw = spark.read.option("multiLine", True).json(paths)
    if "data" not in raw.columns: 
        print(f"Skipping {collection}: expected data field missing; inspect source contract")
        continue
    records = raw.select(F.explode_outer("data").alias("record")).select("record.*")
    records = flatten_structs(records)
    for field in records.schema.fields:
        if isinstance(field.dataType, T.ArrayType):
            records = records.withColumn(field.name, F.to_json(F.col(field.name)))
    records = records.withColumn("_source_collection", F.lit(collection)).withColumn("_processed_at_utc", F.current_timestamp())
    table = f"{CATALOG}.{SILVER_SCHEMA}.{collection}"
    records.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table)
    silver_tables[collection] = table
    print(collection, records.count(), table)

## 5. EDA and data quality
Inspect schemas, row counts, nulls and duplicate keys before creating business aggregates.

In [ ]:
for collection, table in silver_tables.items():
    df = spark.table(table)
    print(f"\\n### {collection}: {df.count():,} rows")
    df.printSchema()
    display(df.limit(5))

def profile_table(table_name):
    df = spark.table(table_name)
    expressions = [F.count(F.lit(1)).alias("row_count")]
    expressions += [F.sum(F.col(c).isNull().cast("long")).alias(f"{c}_nulls") for c in df.columns]
    return df.agg(*expressions)

for collection, table in silver_tables.items():
    print("Quality profile:", collection)
    display(profile_table(table))

## 6. Gold — dimensional and analytical tables
The following section is deliberately defensive: source API field names can evolve. Inspect Silver schemas and adjust the candidate field names if required.

In [ ]:
def first_existing(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

def save_gold(df, name):
    table = f"{CATALOG}.{GOLD_SCHEMA}.{name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table)
    print("Created", table, "rows:", df.count())
    return table

if "products" in silver_tables:
    products = spark.table(silver_tables["products"])
    save_gold(products, "dim_product")
if "customers" in silver_tables:
    customers = spark.table(silver_tables["customers"])
    save_gold(customers, "dim_customer")

if "order_items" in silver_tables:
    oi = spark.table(silver_tables["order_items"])
    qty = first_existing(oi, ["quantity", "qty"])
    price = first_existing(oi, ["unit_price", "price", "selling_price"])
    order_id = first_existing(oi, ["order_id", "orderId"])
    product_id = first_existing(oi, ["product_id", "productId"])
    if qty and price and order_id:
        fact = oi.withColumn("quantity", F.col(qty).cast("double")).withColumn("unit_price", F.col(price).cast("double"))
        fact = fact.withColumn("line_revenue", F.col("quantity") * F.col("unit_price"))
        save_gold(fact, "fact_order_items")
        save_gold(fact.groupBy(order_id).agg(F.sum("line_revenue").alias("order_revenue"), F.sum("quantity").alias("units")), "agg_order_performance")
        if product_id:
            save_gold(fact.groupBy(product_id).agg(F.sum("line_revenue").alias("revenue"),F.sum("quantity").alias("units"),F.countDistinct(order_id).alias("orders")), "agg_product_performance")
    else:
        print("Could not identify required order item fields; inspect order_items schema and update candidates.")

if "reviews" in silver_tables:
    reviews = spark.table(silver_tables["reviews"])
    rating = first_existing(reviews, ["rating", "score", "stars"])
    product_id = first_existing(reviews, ["product_id", "productId"])
    if rating and product_id:
        save_gold(reviews.groupBy(product_id).agg(F.count("*").alias("review_count"),F.avg(F.col(rating).cast("double")).alias("avg_rating")), "agg_review_performance")

if "inventory" in silver_tables:
    save_gold(spark.table(silver_tables["inventory"]), "agg_inventory_health")

## 7. Gold KPI queries and Power BI hand-off
Use these as starting points. Confirm grain, cancellations, refunds, discounts, tax and shipping rules before publishing financial KPIs.

In [ ]:
gold_tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}")
display(gold_tables)

if spark.catalog.tableExists(f"{CATALOG}.{GOLD_SCHEMA}.fact_order_items"):
    display(spark.sql(f\"\"\"SELECT COUNT(DISTINCT order_id) AS orders, SUM(quantity) AS units, SUM(line_revenue) AS gross_line_revenue, SUM(line_revenue)/COUNT(DISTINCT order_id) AS revenue_per_order FROM {CATALOG}.{GOLD_SCHEMA}.fact_order_items\"\"\"))

## 8. Power BI model and next steps

Recommended model: `fact_order_items` at order-line grain, with `dim_product`, `dim_customer` and a date dimension related by the relevant keys. Create measures for revenue, units, distinct orders and average order value. Build executive, product/category, customer, inventory and review pages.

**Before publishing:** validate the live API response, inspect all source columns, test joins and key uniqueness, document KPI definitions, and execute the complete notebook in your own Databricks workspace.

## 9. Git version-control workflow

Use feature branches and pull requests rather than committing directly to `main`:

```bash
git clone https://github.com/kartikch4/retail_intelligence.git
cd retail_intelligence
git checkout -b feature/add-customer-segmentation
git add notebooks/retail_intelligence_end_to_end.ipynb
git commit -m "Add customer segmentation analysis"
git push -u origin feature/add-customer-segmentation
```

Never commit API secrets, workspace credentials or real customer data.